# CUDA 编译运行模板 (Colab / T4)

**用法**:
1. 菜单 `代码执行程序` → `更改运行时类型` → 硬件加速器选 **T4 GPU**。
2. 从上到下依次运行各 cell。
3. 第 2 步会 clone/pull `cuda-learning` 仓库拿到最新代码;第 3 步指定要编译的源文件路径(仓库里的实际文件,比如 `week1/vecadd.cu`),不需要手动复制粘贴代码。

> 本机(Mac Mini)不能跑 nvcc,所有编译运行都在这里做。编译目标 `-arch=sm_75`(T4, Turing)。
>
> 本地改完代码后记得先 `git push`,再回到这里重新跑第 2 步的 cell 拉取最新版本,避免跑到本地已经改过、但仓库里还是旧版的代码。

## 1. 确认 GPU 环境

In [ ]:
!nvidia-smi
!nvcc --version

## 2. 拉取仓库

从 GitHub 仓库 clone/pull,始终拿到与本地仓库同步的最新版本。

In [ ]:
import os

repo_dir = "cuda-learning"
if os.path.isdir(repo_dir):
    !git -C {repo_dir} pull
else:
    !git clone https://github.com/saintlion/cuda-learning.git {repo_dir}

## 3. 选择要编译的源文件

指向仓库里你要跑的那个 `.cu` 文件即可,`common.h` 就在同一目录下,nvcc 会自动找到,不需要额外拷贝或加 `-I`。

In [ ]:
SOURCE_FILE = f"{repo_dir}/week1/vecadd.cu"  # 改成你要跑的练习文件路径

## 4. 编译

`-arch=sm_75` 对应 T4 (Turing)。如果要在 GTX 1050 Ti (sm_61) 上跑,注意别用 Turing+ 独有特性。
`-lineinfo` 让 profiler 能把指令映射回源码行;`-O2` 常规优化。

In [ ]:
!nvcc -arch=sm_75 -O2 -lineinfo {SOURCE_FILE} -o main 2>&1

## 5. 运行

In [ ]:
!./main

## 6.(可选)compute-sanitizer 查内存越界 / 竞态

类比嵌入式里的 Valgrind / 硬件 MPU 报错。kernel 越界、未初始化读、竞态在这里能抓到。

In [ ]:
!compute-sanitizer --tool memcheck ./main